# Демонстрация биннинга (`binning/`)

Пакет `binning/` — два самостоятельных класса в стиле scikit-learn (`fit` / `transform`):

- **`NumBinner`** — числовая фича: квантильные интервалы + «точки» (супервстречаемые значения) + слияние редких бинов;
- **`CatBinner`** — категориальная фича: частые категории сохраняются, редкие → `other`, пропуски → `missing`.

Биннинг **без таргета** (unsupervised) и ничего не знает про PSI — пригоден и для PSI, и для WoE. Тип фичи задаётся **явно** (выбором класса), а не по dtype.

Данные генерируются прямо в ноутбуке (numpy) — примеры самодостаточны.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd
from binning import NumBinner, CatBinner

rng = np.random.default_rng(0)
pd.set_option('display.width', 160)
print('loaded')

loaded


## 1. `NumBinner` — числовой биннер

Параметры конструктора:

| Параметр | Назначение | По умолчанию |
|---|---|---|
| `n_bins` | число квантильных интервалов непрерывного фона | `10` |
| `min_frequency` | минимальный размер бина; меньшие сливаются с соседом | `"auto"` |
| `point_share` | порог доли, при котором значение становится «точкой» (спайк) | `0.10` |

`fit(X)` учит границы, `transform(X)` размечает. Результат — упорядоченная `Categorical`-Series: интервал = `pd.Interval`, точка = само число, `other` / `missing` — служебные строки.

### 1.1 `n_bins` — разрешение квантильной сетки

Непрерывная фича делится на `n_bins` квантильных интервалов (примерно равных по числу наблюдений). Чем больше `n_bins`, тем мельче интервалы.

In [2]:
x = pd.Series(rng.lognormal(mean=10, sigma=0.5, size=5000))
for n in (4, 10):
    cats = NumBinner(n_bins=n).fit_transform(x).cat.categories
    print(f'n_bins={n:2d} -> {len(cats)} бинов')
    print('   ', [str(c) for c in cats])

n_bins= 4 -> 5 бинов
    ['(-inf, 15640.284]', '(15640.284, 21724.015]', '(21724.015, 30919.778]', '(30919.778, inf]', 'missing']
n_bins=10 -> 11 бинов
    ['(-inf, 11753.629]', '(11753.629, 14438.063]', '(14438.063, 16774.741]', '(16774.741, 19157.08]', '(19157.08, 21724.015]', '(21724.015, 25004.355]', '(25004.355, 28664.443]', '(28664.443, 33636.947]', '(33636.947, 42062.634]', '(42062.634, inf]', 'missing']


### 1.2 `point_share` — выделение «точек» (спайков)

Если доля одного значения ≥ `point_share`, оно становится **отдельным бином-точкой**, а не растворяется в интервале. Нужно для супервстречаемых значений: ноль-инфляция, дефолтные заполнители, «магические» константы. Точка проходит ещё и порог `min_frequency` (бин не меньше минимального).

In [3]:
# 40% значений — ровно 0.0 (спайк), остальное непрерывно
x = pd.Series(np.where(rng.random(5000) < 0.4, 0.0, rng.lognormal(8, 0.5, 5000)))
for ps in (0.10, 0.50):
    cats = list(NumBinner(point_share=ps).fit_transform(x).cat.categories)
    is_point = any(isinstance(c, float) for c in cats)
    print(f'point_share={ps:.2f} -> 0.0 отдельная точка? {is_point}  (спайк = 40%)')

point_share=0.10 -> 0.0 отдельная точка? True  (спайк = 40%)
point_share=0.50 -> 0.0 отдельная точка? False  (спайк = 40%)


### 1.3 `min_frequency` — порог редкого бина

Бины меньше порога **сливаются** с соседним. Значения:

- `"auto"` — `clip(0.01·N, 20, 1000)` (доля выборки с полом/потолком);
- `float` из `(0, 1)` — доля от размера выборки;
- `int ≥ 1` — абсолютное число наблюдений;
- `None` / `0` — слияние выключено.

Порог считается от **всей** обученной выборки и применяется единообразно ко всем бинам.

In [4]:
x = pd.Series(rng.lognormal(10, 0.5, 3000))
for mf in (None, 0.15, 0.30):
    cats = NumBinner(n_bins=10, min_frequency=mf).fit_transform(x).cat.categories
    intervals = [c for c in cats if isinstance(c, pd.Interval)]
    print(f'min_frequency={str(mf):5} -> {len(intervals)} интервалов')

min_frequency=None  -> 10 интервалов
min_frequency=0.15  -> 5 интервалов
min_frequency=0.3   -> 3 интервалов


### 1.4 Малая кардинальность — дискретный режим

Если уникальных значений ≤ `n_bins`, квантили не строятся: каждое частое значение получает свой бин (точку), редкие уйдут в `other` при `transform`. Так число-кодированные категории (например, 1..5 продуктов) не дробятся ошибочно.

In [5]:
x = pd.Series(rng.choice([1, 2, 3, 4, 5], size=3000, p=[.4, .3, .18, .08, .04]))
out = NumBinner(n_bins=10).fit_transform(x)
print('бины:', list(out.cat.categories))
out.value_counts().sort_index()

бины: [1.0, 2.0, 3.0, 4.0, 5.0, 'other', 'missing']


1.0        1141
2.0         921
3.0         590
4.0         241
5.0         107
other         0
missing       0
Name: count, dtype: int64

### 1.5 `fit` на train, `transform` на новых данных

Границы учатся один раз на обучающей выборке и применяются к любым новым данным. Значения вне обученного диапазона попадают в крайние интервалы `(-inf, …]` / `(…, inf]`, не ломая разметку.

In [6]:
binner = NumBinner(n_bins=5).fit(pd.Series(rng.lognormal(10, 0.4, 3000)))
new = pd.Series([0.0, 1e9])  # значения за пределами train
print(binner.transform(new).tolist())

[Interval(-inf, 15690.009, closed='right'), Interval(30569.362, inf, closed='right')]


## 2. `CatBinner` — категориальный биннер

Единственный параметр — `min_frequency` (та же семантика порога). При `fit` сохраняются категории с числом наблюдений ≥ порога (по убыванию частоты). При `transform`:

- редкие и **новые** (не виденные на `fit`) категории → `other`;
- пропуски → `missing`.

In [7]:
pool = ['web', 'mobile', 'branch', 'call_center', 'partner']
pool += [f'rare_{i:02d}' for i in range(20)]
p = [.35, .25, .18, .04, .03] + [.15 / 20] * 20
x = pd.Series(rng.choice(pool, size=4000, p=p))

for mf in ('auto', 0.05):
    kept = CatBinner(min_frequency=mf).fit(x).points_
    print(f'min_frequency={str(mf):5} -> сохранено {len(kept)}: {kept}')

min_frequency=auto  -> сохранено 6: ['web', 'mobile', 'branch', 'call_center', 'partner', 'rare_13']
min_frequency=0.05  -> сохранено 3: ['web', 'mobile', 'branch']


In [8]:
# новая категория и пропуск на transform
binner = CatBinner(min_frequency=0.05).fit(x)
probe = pd.Series(['web', 'rare_03', 'BRAND_NEW', None])
print(binner.transform(probe).tolist())

['web', 'other', 'other', 'missing']


## 3. Структура результата

`transform` возвращает упорядоченную `Categorical`-Series. Идентичность бина:

- интервал → `pd.Interval` (right-closed `(a, b]`; у интервала, упёртого в точку, правая граница открыта);
- точка → само число (`float`);
- служебные → строки `other` / `missing`.

Выученные атрибуты: `points_` (точки / частые категории), `edges_` (границы интервалов, у `NumBinner`), `bins_` (итоговые категории после `transform`). Результат напрямую идёт в расчёт PSI или WoE.

In [9]:
b = NumBinner(n_bins=6).fit(pd.Series(rng.lognormal(10, 0.5, 3000)))
print('points_:', b.points_)
print('edges_ :', b.edges_)

points_: []
edges_ : [          -inf 13313.5290595  17668.19641233 21804.14283116
 27475.44919906 35534.60849096            inf]
